# ROUGE-Entailment Correlation

Addresses Examiner 2's comment: *"Bagaimana kedudukan ROUGE dan entailment? Apakah anda hanya lebih mengutamakan entailment?"*

This notebook computes the per-document correlation between ROUGE-1 and the NLI entailment score, as empirical evidence that the two metrics measure different dimensions of summary quality (lexical overlap vs. factual grounding) — supporting the "ROUGE and entailment are not a fixed hierarchy" argument drafted for thesis subbab 4.4.3 / 4.5.

**Prerequisite bug fix reused from `05_significance_test.ipynb`:** `predictions_baseline.jsonl`'s stored `entailment_score` field is always `0.0` due to a pipeline bug (the baseline branch in `01_training.ipynb` never calls the NLI scorer). This notebook recomputes baseline entailment via a fresh batched NLI forward pass over the already-generated `document`/`generated_summary` pairs — no summary regeneration needed. If you have already run `05_significance_test.ipynb` in the same Kaggle session, you can skip Section 3 and reuse its `baseline_entailment` dict directly instead of recomputing.

## 1. Setup

In [ ]:
!pip install -q transformers rouge-score scipy numpy

import json
import statistics
from pathlib import Path

import numpy as np
import torch
from scipy import stats
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
np.random.seed(SEED)
print("GPU available:", torch.cuda.is_available())

## 2. Load Predictions

In [ ]:
import glob

# Flat dataset attached via Add Input (see 06_claim_level_hallucination.ipynb for the same
# setup). Falls back to a recursive glob if Kaggle mounted it under a different slug.
PREDICTIONS_BASELINE_FILE = "/kaggle/input/datasets/madedwikibudilaksana/prediction-21092026/predictions_baseline.jsonl"
PREDICTIONS_NLI_FILE = "/kaggle/input/datasets/madedwikibudilaksana/prediction-21092026/predictions_nli.jsonl"

if not Path(PREDICTIONS_BASELINE_FILE).exists():
    hits = glob.glob("/kaggle/input/**/predictions_baseline.jsonl", recursive=True)
    if hits:
        PREDICTIONS_BASELINE_FILE = hits[0]
if not Path(PREDICTIONS_NLI_FILE).exists():
    hits = glob.glob("/kaggle/input/**/predictions_nli.jsonl", recursive=True)
    if hits:
        PREDICTIONS_NLI_FILE = hits[0]

if not Path(PREDICTIONS_BASELINE_FILE).exists() or not Path(PREDICTIONS_NLI_FILE).exists():
    print("Not found. Contents of /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        print(" ", p)
    raise FileNotFoundError("predictions_baseline.jsonl / predictions_nli.jsonl not found under /kaggle/input.")

print("Baseline file:", PREDICTIONS_BASELINE_FILE)
print("NLI file:", PREDICTIONS_NLI_FILE)

def load_by_id(path):
    rows = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                rows[row["id"]] = row
    return rows

baseline_rows = load_by_id(PREDICTIONS_BASELINE_FILE)
nli_rows = load_by_id(PREDICTIONS_NLI_FILE)
shared_ids = sorted(set(baseline_rows.keys()) & set(nli_rows.keys()))
print(f"Baseline: {len(baseline_rows)} | NLI: {len(nli_rows)} | Shared (paired): {len(shared_ids)}")

## 3. Fix Baseline Entailment Bug (recompute via fresh NLI pass)

In [ ]:
NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32 if DEVICE == "cuda" else 8

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(DEVICE).eval()
id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
ent_idx = next(i for i, l in id2label.items() if "entail" in l)

@torch.inference_mode()
def score_entailment_batch(premises, hypotheses):
    enc = nli_tokenizer(premises, hypotheses, truncation=True, max_length=512, padding=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    probs = torch.softmax(nli_model(**enc).logits, dim=-1)
    return probs[:, ent_idx].tolist()

baseline_entailment = {}
for i in range(0, len(shared_ids), BATCH_SIZE):
    batch_ids = shared_ids[i:i + BATCH_SIZE]
    premises = [baseline_rows[sid]["document"] for sid in batch_ids]
    hypotheses = [baseline_rows[sid]["generated_summary"] for sid in batch_ids]
    ents = score_entailment_batch(premises, hypotheses)
    for sid, ent in zip(batch_ids, ents):
        baseline_entailment[sid] = ent
    if (i + len(batch_ids)) % (BATCH_SIZE * 20) < BATCH_SIZE:
        print(f"Recomputed baseline entailment: {i + len(batch_ids)}/{len(shared_ids)}", flush=True)

print(f"\nRecomputed baseline avg entailment: {statistics.mean(baseline_entailment.values()):.4f} "
      f"(should be close to 0.3100, per Table IV / main_results.json)")

## 4. Per-Document ROUGE-1 and Correlation with Entailment

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

rouge1_baseline, entail_baseline = [], []
rouge1_nli, entail_nli = [], []

for i, sid in enumerate(shared_ids):
    b, n = baseline_rows[sid], nli_rows[sid]
    rouge1_baseline.append(scorer.score(b["reference_summary"], b["generated_summary"])["rouge1"].fmeasure)
    entail_baseline.append(baseline_entailment[sid])
    rouge1_nli.append(scorer.score(n["reference_summary"], n["generated_summary"])["rouge1"].fmeasure)
    entail_nli.append(n["entailment_score"])
    if (i + 1) % 2000 == 0:
        print(f"Scored {i+1}/{len(shared_ids)}")

def report_correlation(rouge_scores, entail_scores, label):
    pearson_r, pearson_p = stats.pearsonr(rouge_scores, entail_scores)
    spearman_r, spearman_p = stats.spearmanr(rouge_scores, entail_scores)
    print(f"[{label}] Pearson r = {pearson_r:.4f} (p={pearson_p:.2e}) | "
          f"Spearman rho = {spearman_r:.4f} (p={spearman_p:.2e}) | n={len(rouge_scores)}")
    return {
        "label": label, "n": len(rouge_scores),
        "pearson_r": round(float(pearson_r), 4), "pearson_p": float(pearson_p),
        "spearman_rho": round(float(spearman_r), 4), "spearman_p": float(spearman_p),
    }

print("=== Correlation between ROUGE-1 and Entailment (per document) ===")
corr_baseline = report_correlation(rouge1_baseline, entail_baseline, "baseline")
corr_nli = report_correlation(rouge1_nli, entail_nli, "BART+NLI")

# Pooled across both conditions (2x n points) — gives a larger-sample view of whether
# "higher ROUGE" and "higher entailment" move together in general, regardless of condition.
pooled_rouge = rouge1_baseline + rouge1_nli
pooled_entail = entail_baseline + entail_nli
corr_pooled = report_correlation(pooled_rouge, pooled_entail, "pooled (baseline+NLI)")

## 5. Save Results

In [ ]:
output_path = Path("./results/rouge_entailment_correlation_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump({"baseline": corr_baseline, "nli": corr_nli, "pooled": corr_pooled}, f, indent=2)
print(f"Saved to {output_path}")

## Narasi untuk disalin ke tesis (draf, sesuaikan dengan angka aktual hasil run)

Contoh kalimat untuk subbab 4.4.3 / 4.5, mengisi Examiner 2 #2:

> Untuk memperkuat argumen bahwa ROUGE dan skor *entailment* mengukur dimensi kualitas yang berbeda, dilakukan analisis korelasi antara ROUGE-1 dan skor *entailment* pada level dokumen (n = **[ISI]**). Korelasi Pearson yang diperoleh sebesar **[ISI]** (p **[ISI]**) dan korelasi Spearman sebesar **[ISI]** (p **[ISI]**), yang tergolong **[lemah/sedang, sesuaikan interpretasi]** — menunjukkan bahwa ringkasan dengan ROUGE tinggi tidak serta-merta memiliki skor *entailment* tinggi, dan sebaliknya. Temuan ini konsisten dengan hasil pada Tabel 4.10 (perbandingan strategi *re-ranking*), di mana *re-ranking* berbasis kemiripan semantik menghasilkan ROUGE-1 tertinggi namun skor *entailment* yang jauh lebih rendah dibandingkan *re-ranking* berbasis NLI. Dengan demikian, ROUGE dan *entailment* pada penelitian ini diperlakukan sebagai dua dimensi yang saling melengkapi — kelancaran/tumpang-tindih leksikal versus konsistensi faktual — bukan sebagai satu metrik yang lebih diutamakan secara mutlak di atas yang lain (lihat juga subbab 4.4.3 mengenai kriteria pemilihan bobot α).